## 실제 에이전트 프로젝트로 발전시키기

지금까지 배운 것들을 이용해서 실제 agent와 같이 파이썬 프로젝트를 구성해 보도록 하겠습니다.

우선 저희가 만들 에이전트의 구조는 아래와 같습니다!

![에이전트 구조](basic_agent.png)

그냥 하면 어려울 수 있으니, '노트북에서 먼저 코드를 테스트 -> 실제 프로젝트 코드에 옮기기' 순서로 진행할게요!

한 단계씩 차근차근 해봅시다!

1. `tools.py`에 들어갈 것들을 작성 및 테스트
2. `middleware.py`에 들어갈 것들을 작성 및 테스트
3. `agent.md`에 들어갈 것들을 작성 및 테스트
4. `agent.py`에 들어갈 것들을 작성 및 테스트

### 1. tool 정의하기

우선, 저희가 만들 에이전트에 필요한 도구가 무엇인가요?

해당 도구들을 하나씩 구현해 봅시다!

1. 계산기

In [1]:
from langchain.tools import tool

@tool
def calculator(a: int, b: int, operator: str) -> float:
    """
    2개의 정수 a, b를 입력 받으면 연산자(operator) string에 
    해당하는 연산을 수행하고 그 결과를 반환하는 함수.
    operator에는 'plus', 'minus', 'multiply', 'divide' 4가지만 들어갈 수 있다.
    """
    if operator == "plus":
        return a + b
    if operator == "minus":
        return a - b
    if operator == "multiply":
        return a * b
    if operator == "divide":
        return a / b

c:\Users\rando\anaconda3\envs\langchain\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


2. 위키 백과 검색기

In [2]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

wikipedia_search = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500)
)

3. 사용자 정보 저장하고 불러오기

In [3]:
import json
from langchain_core.runnables import RunnableConfig

@tool
def save_user_info(name: str, language: str, config: RunnableConfig) -> str:
    """사용자 정보를 저장합니다."""
    thread_id = config["configurable"]["thread_id"]
    
    try:
        with open("user_profile.json", "r") as f:
            data = json.load(f)
    except FileNotFoundError:
        data = {}
    
    data[thread_id] = {"user_name": name, "prefer_language": language}
    
    with open("user_profile.json", "w") as f:
        json.dump(data, f, ensure_ascii=False)
    
    return f"{name}님 정보를 저장했습니다."

@tool
def load_user_info(thread_id: str) -> str:
    """사용자 정보를 불러옵니다."""
    try:
        with open("user_profile.json", "r") as f:
            data = json.load(f)
        profile = data.get(thread_id)
        if not profile:
            return "저장된 정보가 없습니다."
        return f"이름: {profile['user_name']}, 선호 언어: {profile['prefer_language']}"
    except FileNotFoundError:
        return "저장된 정보가 없습니다."

구현한 툴이 제대로 작동하는지 테스트를 해보겠습니다.

In [4]:
# 계산기 테스트

result = calculator.invoke({"a": 10, "b": 5, "operator": "multiply"})
print(result)

50


In [5]:
# 위키 테스트

result = wikipedia_search.invoke({"query": "하마"})
print(result)

Page: Apache Hama
Summary: Apache Hama  is a distributed computing framework based on bulk synchronous parallel computing techniques for massive scientific computations e.g., matrix, graph and network algorithms. Originally a sub-project of Hadoop, it became an Apache Software Foundation top level project in 2012. It was created by Edward J. Yoon, who named it (short for "Hadoop Matrix Algebra"), and Hama also means hippopotamus in Yoon's native Korean language (하마), following the trend of namin


결과가 잘 나온다면 위에서 구현한 `calculator`와 `wiki_search` tool들을 `tools.py`에 복붙해 봅시다!

(주의 : import 문도 함께 붙여 넣어야 합니다!)

### 2. middleware 정의하기

먼저 저희가 사용할 middleware들이 뭔지 다시 생각해 볼까요?

그리고 하나씩 구현해 보면 되겠습니다!

In [6]:
from langchain.agents.middleware import wrap_tool_call

@wrap_tool_call
def logging_middleware(request, handler): # request와 handler 2개의 인자를 반드시 입력 받아야 합니다!
    # request : 도구 호출 정보
    # handler : 실제 도구 실행 함수
    tool_name = request.tool_call["name"] # tool 이름
    tool_args = request.tool_call["args"] # tool의 입력 인자
    print(f"[호출] 도구: {tool_name} | 인수: {tool_args}")
    try:
        result = handler(request)
        print(f"[도구 호출 성공]")
        return result
    except Exception as e:
        print(f"[도구 호출 실패]")
        raise

요약 모듈은 llm api의 호출을 필요로 하기 때문에 api_key 설정을 먼저 해줘야 합니다.

In [8]:
from dotenv import load_dotenv

# .env 파일로부터 api_key를 불러오는 코드
load_dotenv(dotenv_path=".env")

True

In [9]:
from langchain.agents.middleware import SummarizationMiddleware, ContextEditingMiddleware, ClearToolUsesEdit

context_trimmer = ContextEditingMiddleware(
    edits=[
        ClearToolUsesEdit(
            trigger=2000,
            keep=4,
        )
    ]
)

text_summarizer = SummarizationMiddleware(
    model="gpt-5.4-mini",
    trigger=("tokens", 1000),
    keep=("messages", 2)
)

미들웨어는 독립적으로 테스트하기가 어렵습니다.

일단 여기선 문제가 없다고 생각하고(미리 작성한 코드들을 그대로 썼다면 문제가 없을 겁니다.) 넘어가도록 하겠습니다.

실제로는 꼭 테스트를 해보고 넘어가야 합니다! 테스트 하는 방법은 : 

1. 로깅하기 : 앞서 노트북에서 연습했던 것처럼 미들웨어가 실행됐어야 할 상황에 실행됐는지 직접 print()로 찍어보면서 확인합니다.
2. 테스트 코드 따로 작성하기 : 미들웨어 테스트 코드를 따로 작성해서 해당 코드를 실행해서 미들웨어가 잘 작동하는지 확인하는 방법입니다.
3. LangSmith 활용하기 : langsmith는 에이전트가 실행되는 구조와 흐름을 모니터링 해주는 도구입니다. 지금은 다루지 않지만 실무에서 많이 사용하는 도구 이므로 알아두면 좋습니다.

마찬가지로 위에서 구현한 middleware 코드들을 middleware.py로 옮기면 됩니다.

### 3. 시스템 프롬프트 작성

이번엔 시스템 프롬프트를 작성해 보겠습니다.

아까 했던 `agents.md`를 참고해서 나만의 시스템 프롬프트를 잘 작성해 봅시다.

**꼭 들어가야 하는 내용** : 꼭 도구를 사용하도록 유도해야 합니다.

해당 내용은 바로 `agents.md`에 작성하면 되겠습니다.

### 4. 에이전트 만들기

이번엔 지금까지 작성한 tools, middleware, agents.md의 내용들을 기반으로 에이전트를 만들어 보겠습니다.

`create_agent()`를 사용하면 됐죠?

꼭 메모리 기능도 잊지 말고 추가하여 구현해 줍시다!

<에이전트 세팅하기>

1. .env 파일로부터 api key 불러오고 모델 설정하기

In [10]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# .env 파일로부터 api_key를 불러오는 코드
load_dotenv(dotenv_path=".env")

# 모델 이름
model_name = "gpt-5.4-mini"

# LLM 정의
model = init_chat_model(
    model=model_name,
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0.7,
)

2. agent.md의 시스템 프롬프트를 불러오는 코드 작성

In [12]:
with open("agent/agents.md", encoding="utf-8") as f:
    system_prompt = f.read()

4. create_agent()로 에이전트로 변환 (InMemorySaver 잊지 말기!)

In [13]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model,
    tools=[calculator, wikipedia_search, save_user_info, load_user_info],
    middleware=[logging_middleware, context_trimmer, text_summarizer],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

4. agent 테스트해보기

In [15]:
response = agent.invoke(
    {"messages": {"role": "user", "content": "독수리와 매의 차이가 뭐야?"}},
    {"configurable": {"thread_id": "1"}}
)

print(response['messages'][-1].content)

[호출] 도구: wikipedia | 인수: {'query': 'Eagle Hawk difference birds of prey'}
[성공] 결과: Page: Mountain hawk-eagle
Summary: The mountain hawk-eagle (Nisaetus nipalensis) or Hodgson's hawk-eagle, is a large bird of prey native to Asia. The latter name is in reference to the naturalist, Brian Houghton Hodgson, who described the species after collecting one himself in the Himalayas. A less widely recognized common English name is the feather-toed eagle. Like all eagles, it is in the family Accipitridae. Its feathered tarsus marks this species as a member of the subfamily Aquilinae. It 
[호출] 도구: wikipedia | 인수: {'query': 'Eagle bird of prey hawk bird of prey classification'}
[성공] 결과: Page: Bird of prey
Summary: Birds of prey or predatory birds, also known as raptors, are hypercarnivorous species of bird that actively hunt and feed on other vertebrates, mainly mammals, reptiles, and smaller birds. In addition to speed and strength, these predators have keen eyesight for detecting prey from a dist

그럼 이 코드들도 agent.py로 옮겨주면 되는데... 여기선 몇 가지 주의해야 할 점이 있어요.

주의해야 할 내용들은 `agent.py`에 적어놓을게요.

그럼 이제 필요한 재료들은 모두 모였습니다. 이제 이렇게 만든 agent를 실행할 수 있는 코드를 `main.py`에 작성하면 됩니다!

`main.py`에서 같이 보시죠.